# Session 6 — Multi-Case Automation and Reproducible CFD Reports

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

Real projects rarely involve one CFD run — they involve a sweep of cases (Reynolds number, angle of attack, design variants). This session automates the Session 1–5 workflow across a case sweep and produces a reproducible, auditable summary report.

## Learning outcomes
- Structure a parameter sweep as reusable functions, not copy-pasted cells.
- Run automated data-quality checks across every case in a sweep.
- Tabulate and compare results across cases.
- Produce a reproducible report with explicit pass/fail quality flags.

## Using your own Fluent or CSV data
This notebook uses synthetic data so you can run every cell immediately without a CFD license. When you are ready to use your own results, export a CSV from Fluent (or any solver) with coordinates, variable names, units, operating conditions, and a case identifier, then replace the synthetic-data cell below with:

```python
df = pd.read_csv("your_export.csv")
```

Map your solver's column names to the ones used in this notebook before continuing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

rng = np.random.default_rng(41)
print("Environment ready.")

## 1. Defining the sweep

Reproducibility starts with parameters defined once, in one place — never re-typed inside a loop or hidden inside a plot call.

In [ ]:
sweep_cases = pd.DataFrame({
    "case_id": ["case_A", "case_B", "case_C", "case_D"],
    "reynolds_number": [5.0e4, 1.0e5, 2.0e5, 4.0e5],
    "angle_of_attack_deg": [0.0, 4.0, 8.0, 12.0],
})
U_inf_mps = 20.0
rho_kgpm3 = 1.225
chord_m = 0.15
sweep_cases

## 2. A reusable case-processing function

Wrap the Session 1 quality checks and a synthetic lift/drag calculation into one function so every case is processed identically — this is what makes the sweep reproducible.

In [ ]:
def run_synthetic_case(reynolds_number, alpha_deg, rng):
    """Return a synthetic (Cl, Cd, quality_flags) for one case, mimicking post-processed CFD output."""
    alpha_rad = np.deg2rad(alpha_deg)
    Cl = 2 * np.pi * alpha_rad * (1 - 0.05 * np.log10(reynolds_number / 1e5).clip(min=0)) + rng.normal(0, 0.01)
    Cd0 = 0.012 + 2e-8 * (5e5 - reynolds_number) / 1e5
    Cd = max(Cd0, 0.005) + 0.02 * alpha_rad ** 2 + rng.normal(0, 0.0005)

    residual_final = 10 ** rng.uniform(-8, -5)
    force_drift_pct = rng.uniform(0.1, 2.0)

    quality_flags = {
        "residual_below_1e-6": residual_final < 1e-6,
        "force_drift_below_1pct": force_drift_pct < 1.0,
        "Cl_within_physical_range": -0.5 < Cl < 2.0,
        "Cd_within_physical_range": 0.0 < Cd < 0.5,
    }
    return Cl, Cd, quality_flags, residual_final, force_drift_pct

# Smoke-test on one case before running the full sweep
Cl, Cd, flags, resid, drift = run_synthetic_case(1e5, 4.0, rng)
print(f"Cl={Cl:.4f}, Cd={Cd:.4f}, flags={flags}")

### Checkpoint 1 — Read the function, don't just run it
Before proceeding: what would happen to `Cd0` for a Reynolds number of 6e5? Is the `max(Cd0, 0.005)` floor doing anything at that Reynolds number, or only at low Re? Explaining a function you didn't write is part of using automation responsibly.

## 3. Running the sweep with automated quality checks

Apply `run_synthetic_case` to every row of `sweep_cases`, collecting results and quality flags into one results table — no case is hand-processed differently from another.

In [ ]:
results = []
for _, row in sweep_cases.iterrows():
    Cl, Cd, flags, resid, drift = run_synthetic_case(row.reynolds_number, row.angle_of_attack_deg, rng)
    all_pass = all(flags.values())
    results.append({
        "case_id": row.case_id,
        "reynolds_number": row.reynolds_number,
        "angle_of_attack_deg": row.angle_of_attack_deg,
        "Cl": Cl, "Cd": Cd,
        "residual_final": resid,
        "force_drift_pct": drift,
        "quality_pass": all_pass,
        **{f"flag_{k}": v for k, v in flags.items()},
    })

results_df = pd.DataFrame(results)
results_df

### Checkpoint 2 — Find the failing case
At least one case above is likely to fail a quality flag (residual or force-drift thresholds use random draws each run). Filter `results_df` to show only rows where `quality_pass` is `False`, and identify which specific flag(s) failed.

In [ ]:
# TODO: filter results_df to rows where quality_pass is False and print which flag_* columns are False


## 4. Comparing cases

With every case processed identically, compare them directly — e.g., lift and drag vs. angle of attack across the Reynolds sweep.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(results_df.angle_of_attack_deg, results_df.Cl, marker="o")
axes[0].set_xlabel("angle of attack (deg)")
axes[0].set_ylabel("Cl (-)")
axes[0].set_title("Lift vs. angle of attack")

axes[1].plot(results_df.angle_of_attack_deg, results_df.Cd, marker="o", color="tab:orange")
axes[1].set_xlabel("angle of attack (deg)")
axes[1].set_ylabel("Cd (-)")
axes[1].set_title("Drag vs. angle of attack")

plt.tight_layout()
plt.show()

print(f"Cl/Cd ratio by case:")
for _, r in results_df.iterrows():
    print(f"  {r.case_id}: L/D = {r.Cl/r.Cd:6.2f}  (Re={r.reynolds_number:.1e}, alpha={r.angle_of_attack_deg:.1f} deg)")

## 5. A reproducible report

A reproducible report states exactly what was run, when, with what parameters, and which cases passed quality control — so anyone can regenerate or audit it.

In [ ]:
def generate_report(results_df, sweep_params):
    lines = []
    lines.append(f"CFD Sweep Report — generated {datetime.now():%Y-%m-%d %H:%M}")
    lines.append(f"Reference conditions: U_inf = {sweep_params['U_inf_mps']} m/s, "
                 f"rho = {sweep_params['rho_kgpm3']} kg/m^3, chord = {sweep_params['chord_m']} m")
    lines.append(f"Cases run: {len(results_df)}   "
                 f"Passed quality control: {results_df['quality_pass'].sum()} / {len(results_df)}")
    lines.append("")
    for _, r in results_df.iterrows():
        status = "PASS" if r.quality_pass else "FAIL"
        lines.append(f"[{status}] {r.case_id}: Re={r.reynolds_number:.2e}, "
                     f"alpha={r.angle_of_attack_deg:.1f} deg -> "
                     f"Cl={r.Cl:.4f}, Cd={r.Cd:.4f}, L/D={r.Cl/r.Cd:.2f}")
        if not r.quality_pass:
            failed = [k.replace("flag_", "") for k in results_df.columns
                      if k.startswith("flag_") and not r[k]]
            lines.append(f"         failed checks: {', '.join(failed)}")
    return "\n".join(lines)

report_text = generate_report(results_df, {"U_inf_mps": U_inf_mps, "rho_kgpm3": rho_kgpm3, "chord_m": chord_m})
print(report_text)

### Checkpoint 3 — Make it reproducible
Two things are currently missing for full reproducibility: a fixed random seed reference in the report, and the solver/mesh version that would exist for a real case. Add a line to `generate_report` recording the `rng` seed used (`41`, set at the top of this notebook) and one placeholder line for "solver version" that a real workflow would fill in automatically.

In [ ]:
# TODO: extend generate_report (or write a new version) to include the rng seed and a solver-version line


## Graduate/Advanced Extension
Extend the sweep to a full 2D grid of Reynolds number x angle of attack (e.g., 4 Reynolds numbers x 5 angles = 20 cases) using `itertools.product`, re-run `run_synthetic_case` over all combinations, and produce a heatmap of `Cl/Cd` over the (Re, alpha) grid using `ax.pcolormesh` or `ax.contourf`. This is the natural end point of the course: from a single trustworthy CFD export (Session 1) to an automated, quality-checked design-space study (Session 6).

## Exit ticket
In three sentences: name one part of your own post-processing workflow you would now turn into a reusable function instead of copy-pasted cells, state why an automated quality flag is more trustworthy than a manual "eyeball check" across many cases, and describe what "reproducible" means for a CFD report in your own words.

**Course complete.** You now have a working pipeline from raw CFD export to trustworthy, automated, reproducible engineering results.